# E-Methanol Reactor Predictive Surrogate Model

This notebook trains a Machine Learning (Random Forest) surrogate model using the synthetic data generated from our rigorous 1D physics-based reactor model. 

It allows for **instantaneous prediction** of reactor performance (CO2 conversion, Methanol Selectivity, and Space-Time Yield) without needing to solve the complex ODEs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Load the DOE dataset generated by our 1D model
df = pd.read_csv('../outputs/membrane_doe.csv')
print(f"Loaded {len(df)} simulated reactor cases.")
df.head()

## 1. Train the Machine Learning Surrogate Models
We will train a separate Random Forest for each key performance indicator (Target).

In [ ]:
features = [
    'inlet_temperature_k',
    'inlet_pressure_bar',
    'inlet_flow_mol_s',
    'h2_co2_ratio',
    'length_m',
    'water_permeance_mol_m2_s_pa',
    'sweep_water_partial_pressure_bar'
]

targets = [
    'co2_conversion',
    'methanol_selectivity_carbon',
    'methanol_sty_kg_m3cat_h'
]

X = df[features]
y = df[targets]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {}

for target in targets:
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train[target])
    models[target] = rf
    
    # Evaluate
    preds = rf.predict(X_test)
    r2 = r2_score(y_test[target], preds)
    print(f"{target}: R² = {r2:.3f}")


## 2. Interactive Prediction Interface
Input your own reactor parameters to instantly predict the production results.

In [ ]:
def predict_reactor_performance(T_in, P_in, flow, ratio, length, permeance, sweep_P):
    """
    Predicts the reactor performance using the trained ML models.
    """
    input_data = pd.DataFrame([{
        'inlet_temperature_k': T_in,
        'inlet_pressure_bar': P_in,
        'inlet_flow_mol_s': flow,
        'h2_co2_ratio': ratio,
        'length_m': length,
        'water_permeance_mol_m2_s_pa': permeance,
        'sweep_water_partial_pressure_bar': sweep_P
    }])
    
    print("--- ML Predicted Reactor Performance ---")
    for target, model in models.items():
        pred = model.predict(input_data)[0]
        if 'sty' in target:
            print(f"Methanol Production (STY): {pred:.4f} kg/(m³·h)")
        elif 'conversion' in target:
            print(f"CO2 Conversion:            {pred*100:.2f}%")
        elif 'selectivity' in target:
            print(f"Methanol Selectivity:      {pred*100:.2f}%")

# --- TEST THE PREDICTOR HERE ---
predict_reactor_performance(
    T_in=493.15,      # 220 °C
    P_in=50.0,        # bar
    flow=0.015,       # mol/s
    ratio=3.5,        # H2:CO2 ratio
    length=1.5,       # meters
    permeance=1e-7,   # mol/(m²·s·Pa)
    sweep_P=1e-4      # bar
)

## 3. Feature Importance Analysis
Let's see which operating conditions have the biggest impact on Methanol Production.

In [ ]:
target = 'methanol_sty_kg_m3cat_h'
importances = models[target].feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 5))
plt.title("Impact of Operating Conditions on Methanol Production")
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()